# LeetCode 1013 — Partition Array Into Three Parts With Equal Sum bằng Python

**Link đề:** https://leetcode.com/problems/partition-array-into-three-parts-with-equal-sum/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Prefix Sum (tổng tiền tố)

```python
class Solution:
    def canThreePartsEqualSum(self, arr: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: chia mảng thành **3 đoạn liên tiếp, khác rỗng**, tổng bằng nhau.
2. Nắm các công cụ: `sum`, slicing, **prefix sum**, `itertools.accumulate`, `%` và `//`.
3. Viết baseline thử mọi cách cắt (O(n²) với prefix sum) để làm "trọng tài".
4. Rút ra insight: mỗi phần phải có tổng `total / 3` → chỉ cần quét **một lần** và đếm.
5. Hiểu các bẫy: `total` không chia hết cho 3, `target = 0`, phần thứ ba **rỗng**, thứ tự hai điểm cắt.
6. Trace, chứng minh, phân tích độ phức tạp, so sánh 4 cách, làm bài tập biến thể (chia `k` phần).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — cắt mảng thành 3 đoạn có tổng bằng nhau

Cho mảng số nguyên `arr`. Trả về `True` nếu có thể chia `arr` thành **ba phần khác rỗng** có tổng bằng nhau.

Hình thức: tồn tại chỉ số `i + 1 < j` sao cho

```
arr[0] + ... + arr[i]  ==  arr[i+1] + ... + arr[j-1]  ==  arr[j] + ... + arr[n-1]
```

Tức là: phần 1 = `arr[0..i]`, phần 2 = `arr[i+1..j-1]` (khác rỗng vì `i + 1 < j`), phần 3 = `arr[j..n-1]` (khác rỗng vì `j ≤ n-1`).

| `arr` | Output | Giải thích |
|---|:---:|---|
| `[0,2,1,-6,6,-7,9,1,2,0,1]` | `True` | `0+2+1 = -6+6-7+9+1 = 2+0+1 = 3` |
| `[0,2,1,-6,6,7,9,-1,2,0,1]` | `False` | tổng 21, mỗi phần cần 7 nhưng không cắt được |
| `[3,3,6,5,-2,2,5,1,-9,4]` | `True` | `3+3 = 6 = 5-2+2+5+1-9+4` |

**Constraints:** `3 <= arr.length <= 5·10^4`, `-10^4 <= arr[i] <= 10^4`.

Câu hỏi tự đặt:
- Có được **sắp xếp lại** mảng không? (Không — các phần là **đoạn liên tiếp** theo thứ tự gốc.)
- Ba phần bằng nhau thì mỗi phần bằng bao nhiêu? (Gợi ý 1 của đề.)
- Có số âm → tổng tiền tố có thể đi lên đi xuống, không đơn điệu.

In [1]:
arr = [0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]
p1, p2, p3 = arr[0:3], arr[3:8], arr[8:]
print(p1, sum(p1))
print(p2, sum(p2))
print(p3, sum(p3))

[0, 2, 1] 3
[-6, 6, -7, 9, 1] 3
[2, 0, 1] 3


`arr[a:b]` là **slicing**: lấy các phần tử từ chỉ số `a` tới **trước** `b`. `arr[8:]` lấy từ 8 tới hết.

## 2. Nền tảng cần dùng

### 2.1. Mỗi phần bằng `total / 3`

Nếu ba phần cùng tổng `t` thì `total = 3t` → `t = total / 3`. Hệ quả ngay:
- `total % 3 != 0` → **chắc chắn `False`**.
- Ngược lại `t = total // 3` (chia nguyên, giữ kiểu `int`).

In [2]:
for arr in [[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1], [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1], [1, 2, 4]]:
    total = sum(arr)
    print(f"total = {total:>3}, total % 3 = {total % 3}, target = {total // 3 if total % 3 == 0 else '—'}")

total =   9, total % 3 = 0, target = 3
total =  21, total % 3 = 0, target = 7
total =   7, total % 3 = 1, target = —


**Chú ý số âm với `%` và `//` trong Python:** `-7 % 3 = 2`, `-7 // 3 = -3` (làm tròn xuống). Với `total = -9`: `-9 % 3 = 0`, `-9 // 3 = -3` — vẫn đúng. Nên kiểm tra `total % 3 == 0` an toàn cả với tổng âm.

In [3]:
print(-7 % 3, -7 // 3)
print(-9 % 3, -9 // 3)
print(-9 / 3, type(-9 / 3))   # '/' cho float — tránh dùng để so sánh với tổng int

2 -3
0 -3
-3.0 <class 'float'>


### 2.2. Prefix sum (tổng tiền tố)

`prefix[k] = arr[0] + ... + arr[k-1]` (tổng của `k` phần tử đầu), với `prefix[0] = 0`.

Tổng đoạn `arr[a..b]` (gồm cả hai đầu) = `prefix[b+1] - prefix[a]` → tính trong **O(1)** sau khi chuẩn bị prefix trong O(n).

In [4]:
arr = [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]
prefix = [0]
for x in arr:
    prefix.append(prefix[-1] + x)
print("prefix:", prefix)
a, bb = 2, 2
print(f"tổng arr[{a}..{bb}] =", prefix[bb + 1] - prefix[a], "| kiểm tra:", sum(arr[a:bb + 1]))
a, bb = 3, 9
print(f"tổng arr[{a}..{bb}] =", prefix[bb + 1] - prefix[a], "| kiểm tra:", sum(arr[a:bb + 1]))

prefix: [0, 3, 6, 12, 17, 15, 17, 22, 23, 14, 18]
tổng arr[2..2] = 6 | kiểm tra: 6
tổng arr[3..9] = 6 | kiểm tra: 6


`prefix[-1]` là phần tử **cuối** của list (chỉ số âm đếm từ cuối).

### 2.3. `itertools.accumulate` — prefix sum có sẵn

`accumulate(arr)` sinh lần lượt `arr[0]`, `arr[0]+arr[1]`, … (không có số 0 ở đầu). Thêm `initial=0` để có số 0 đầu.

In [5]:
from itertools import accumulate
arr = [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]
print(list(accumulate(arr)))
print(list(accumulate(arr, initial=0)))

[3, 6, 12, 17, 15, 17, 22, 23, 14, 18]
[0, 3, 6, 12, 17, 15, 17, 22, 23, 14, 18]


## 3. Cách trực giác / Baseline

### 3.1. Cách 1 — Thử mọi cặp điểm cắt `(i, j)` và cộng bằng `sum` (O(n³))

Duyệt mọi `i` từ `0` và mọi `j` từ `i + 2` tới `n - 1`, cộng ba đoạn bằng slicing. Dễ hiểu nhất nhưng chậm.

In [6]:
def can_three_bruteforce(arr: list[int]) -> bool:
    n = len(arr)
    for i in range(n):
        for j in range(i + 2, n):
            s1 = sum(arr[:i + 1])
            s2 = sum(arr[i + 1:j])
            s3 = sum(arr[j:])
            if s1 == s2 == s3:
                return True
    return False

print(can_three_bruteforce([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]))   # True
print(can_three_bruteforce([0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1]))   # False
print(can_three_bruteforce([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]))      # True

True
False
True


- `s1 == s2 == s3` là **so sánh chuỗi** của Python, nghĩa là `s1 == s2 and s2 == s3`.
- `range(i + 2, n)` đảm bảo `i + 1 < j` (phần 2 khác rỗng) và `j ≤ n - 1` (phần 3 khác rỗng).

### 3.2. Cách 2 — Thử mọi cặp nhưng dùng prefix sum (O(n²))

Thay ba lần `sum(...)` O(n) bằng hiệu prefix O(1).

In [7]:
def can_three_prefix_pairs(arr: list[int]) -> bool:
    n = len(arr)
    prefix = list(accumulate(arr, initial=0))
    total = prefix[n]
    for i in range(n):
        s1 = prefix[i + 1]
        for j in range(i + 2, n):
            s2 = prefix[j] - prefix[i + 1]
            s3 = total - prefix[j]
            if s1 == s2 == s3:
                return True
    return False

print([can_three_prefix_pairs(a) for a in [[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1],
                                            [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1],
                                            [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]]])

[True, False, True]


## 4. Nút thắt

Với `n = 5·10^4`, O(n²) ≈ `2.5·10^9` / 2 phép → quá chậm.

Lãng phí ở đâu? Ta thử mọi `j` cho mỗi `i`, nhưng thật ra **đã biết trước tổng cần đạt** `t = total / 3`:
- Phần 1 phải có tổng `t` → `prefix[i+1] = t`.
- Phần 1 + phần 2 phải có tổng `2t` → `prefix[j] = 2t`.
- Khi đó phần 3 tự động là `total - 2t = t`.

Vậy bài toán trở thành: **tìm hai vị trí trên dãy prefix, vị trí đầu có giá trị `t`, vị trí sau có giá trị `2t`, và cách nhau đủ xa**.

In [8]:
arr = [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]
total = sum(arr); t = total // 3
pre = list(accumulate(arr))
print("t =", t, "| 2t =", 2 * t)
for idx, p in enumerate(pre):
    mark = "<- t" if p == t else ("<- 2t" if p == 2 * t else "")
    print(f"idx={idx}, arr[idx]={arr[idx]:>2}, prefix tới idx = {p:>2} {mark}")

t = 6 | 2t = 12
idx=0, arr[idx]= 3, prefix tới idx =  3 
idx=1, arr[idx]= 3, prefix tới idx =  6 <- t
idx=2, arr[idx]= 6, prefix tới idx = 12 <- 2t
idx=3, arr[idx]= 5, prefix tới idx = 17 
idx=4, arr[idx]=-2, prefix tới idx = 15 
idx=5, arr[idx]= 2, prefix tới idx = 17 
idx=6, arr[idx]= 5, prefix tới idx = 22 
idx=7, arr[idx]= 1, prefix tới idx = 23 
idx=8, arr[idx]=-9, prefix tới idx = 14 
idx=9, arr[idx]= 4, prefix tới idx = 18 


Ở đây `prefix = t = 6` lần đầu tại `idx = 1` (phần 1 = `[3,3]`), và `prefix = 2t = 12` tại `idx = 2` (phần 2 = `[6]`). Phần 3 = `arr[3:]` có tổng 6 ✓.

## 5. Insight — quét tham lam một lần

**Tham lam:** cắt phần 1 **ngay khi** tổng chạy đạt `t` lần đầu; rồi bắt đầu đếm lại, cắt phần 2 ngay khi đạt `t` lần nữa; nếu **còn ít nhất một phần tử phía sau** thì đúng.

Cài đặt gọn: duyệt, cộng dồn `cur`; mỗi khi `cur == t` thì `parts += 1`, `cur = 0`. Cuối cùng `return parts >= 3`.

Vì sao `>= 3` chứ không `== 3`? Khi `t = 0`, có thể đếm được nhiều hơn 3 lần (ví dụ `[0,0,0,0]` đếm 4) — ta gộp các phần thừa vào phần 3 (tổng các phần thừa là 0 nên không đổi). Với `t ≠ 0` thì sau 3 phần, phần dư có tổng `total - 3t = 0`, gộp vào phần 3 cũng không đổi.

In [9]:
def count_parts(arr):
    total = sum(arr)
    t = total // 3
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == t:
            parts += 1
            cur = 0
    return parts

print(count_parts([0, 0, 0, 0]))                       # 4 phần tổng 0
print(count_parts([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]))   # >= 3

4
3


### 5.1. Tại sao **cắt sớm nhất** là an toàn (không làm mất lời giải)?

Giả sử có lời giải với điểm cắt `(i*, j*)`. Gọi `i₀` = vị trí sớm nhất có `prefix = t` → `i₀ ≤ i*`. Vì `prefix[j*] = 2t` với `j* - 1 > i* ≥ i₀`, phần 2 bắt đầu từ `i₀ + 1` vẫn đạt `2t` tại vị trí `≤ j* - 1`. Lấy vị trí sớm nhất sau `i₀` đạt `2t` → còn sớm hơn hoặc bằng `j* - 1`, nên phần 3 vẫn khác rỗng. Cắt sớm chỉ để lại **nhiều chỗ hơn** cho các phần sau.

## 6. Lời giải chính — mổ xẻ rồi ghép

```python
total = sum(arr)                 # Đoạn 1: tổng
if total % 3 != 0:               # Đoạn 2: loại sớm
    return False
target = total // 3
cur = parts = 0
for x in arr:                    # Đoạn 3: quét, đếm số lần tổng chạy chạm target
    cur += x
    if cur == target:
        parts += 1
        cur = 0
return parts >= 3                # Đoạn 4
```

`cur = parts = 0` là **gán chuỗi**: cả hai biến cùng nhận giá trị 0.

In [10]:
def can_three_greedy(arr: list[int]) -> bool:
    total = sum(arr)
    if total % 3 != 0:
        return False
    target = total // 3
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == target:
            parts += 1
            cur = 0
    return parts >= 3

for arr in [[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1], [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1], [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]]:
    print(arr, "->", can_three_greedy(arr))

[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1] -> True
[0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1] -> False
[3, 3, 6, 5, -2, 2, 5, 1, -9, 4] -> True


### 6.1. Phiên bản "tìm hai điểm cắt" tường minh (dễ chứng minh hơn)

Thay vì đếm, ta tìm rõ `i` (đầu tiên có prefix = `t`) rồi `j` (đầu tiên **sau `i`** có prefix = `2t`) và yêu cầu `j < n - 1` (để phần 3 khác rỗng).

Lưu ý: ở đây `j` là chỉ số **phần tử cuối của phần 2**.

In [11]:
def can_three_two_cuts(arr: list[int]) -> bool:
    total = sum(arr)
    if total % 3 != 0:
        return False
    t = total // 3
    n = len(arr)
    s = 0
    i = 0
    while i < n:                  # tìm điểm cắt 1
        s += arr[i]
        if s == t:
            break
        i += 1
    j = i + 1
    while j < n - 1:              # tìm điểm cắt 2, chừa ít nhất 1 phần tử cho phần 3
        s += arr[j]
        if s == 2 * t:
            return True
        j += 1
    return False

for arr in [[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1], [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1], [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]]:
    print(arr, "->", can_three_two_cuts(arr))

[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1] -> True
[0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1] -> False
[3, 3, 6, 5, -2, 2, 5, 1, -9, 4] -> True


Nếu vòng `while i < n` chạy hết mà không `break` thì `i = n` → `j = n + 1`, vòng 2 không chạy → `False` ✓.

### 6.2. Cách 4 — Hai con trỏ từ hai đầu

Tìm phần 1 từ trái (con trỏ `l`) và phần 3 từ phải (con trỏ `r`) độc lập; cần `l + 1 < r` để phần giữa khác rỗng. Phần giữa tự có tổng `total - 2t = t`.

In [12]:
def can_three_two_pointers(arr: list[int]) -> bool:
    total = sum(arr)
    if total % 3 != 0:
        return False
    t = total // 3
    n = len(arr)
    l, left_sum = 0, arr[0]
    while l < n - 1 and left_sum != t:
        l += 1
        left_sum += arr[l]
    r, right_sum = n - 1, arr[n - 1]
    while r > 0 and right_sum != t:
        r -= 1
        right_sum += arr[r]
    return left_sum == t and right_sum == t and l + 1 < r

for arr in [[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1], [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1], [3, 3, 6, 5, -2, 2, 5, 1, -9, 4]]:
    print(arr, "->", can_three_two_pointers(arr))

[0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1] -> True
[0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1] -> False
[3, 3, 6, 5, -2, 2, 5, 1, -9, 4] -> True


## 7. TRACE từng bước

Ví dụ 1: `arr = [0,2,1,-6,6,-7,9,1,2,0,1]`, `total = 9`, `target = 3`.

| idx | x | cur sau cộng | `cur == 3`? | parts | cur sau xử lý |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | ✗ | 0 | 0 |
| 1 | 2 | 2 | ✗ | 0 | 2 |
| 2 | 1 | 3 | ✓ | 1 | 0 |
| 3 | -6 | -6 | ✗ | 1 | -6 |
| 4 | 6 | 0 | ✗ | 1 | 0 |
| 5 | -7 | -7 | ✗ | 1 | -7 |
| 6 | 9 | 2 | ✗ | 1 | 2 |
| 7 | 1 | 3 | ✓ | 2 | 0 |
| 8 | 2 | 2 | ✗ | 2 | 2 |
| 9 | 0 | 2 | ✗ | 2 | 2 |
| 10 | 1 | 3 | ✓ | 3 | 0 |

`parts = 3 ≥ 3` → `True`. Chú ý idx 4: `cur = 0` không phải target nên **không** cắt.

In [13]:
def trace_greedy(arr):
    total = sum(arr)
    print("arr =", arr, "| total =", total)
    if total % 3 != 0:
        print("total không chia hết cho 3 -> False")
        return False
    target = total // 3
    print("target =", target)
    cur = parts = 0
    for idx, x in enumerate(arr):
        cur += x
        hit = cur == target
        if hit:
            parts += 1
        print(f"  idx={idx:>2} x={x:>3} cur={cur:>3} {'== target -> cắt, parts=' + str(parts) if hit else ''}")
        if hit:
            cur = 0
    print("parts =", parts, "->", parts >= 3)
    return parts >= 3

trace_greedy([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1])

arr = [0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1] | total = 9
target = 3
  idx= 0 x=  0 cur=  0 
  idx= 1 x=  2 cur=  2 
  idx= 2 x=  1 cur=  3 == target -> cắt, parts=1
  idx= 3 x= -6 cur= -6 
  idx= 4 x=  6 cur=  0 
  idx= 5 x= -7 cur= -7 
  idx= 6 x=  9 cur=  2 
  idx= 7 x=  1 cur=  3 == target -> cắt, parts=2
  idx= 8 x=  2 cur=  2 
  idx= 9 x=  0 cur=  2 
  idx=10 x=  1 cur=  3 == target -> cắt, parts=3
parts = 3 -> True


True

### 7.1. Trace ví dụ 2 (False) — tổng chia hết nhưng không cắt được

In [14]:
trace_greedy([0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1])

arr = [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1] | total = 21
target = 7
  idx= 0 x=  0 cur=  0 
  idx= 1 x=  2 cur=  2 
  idx= 2 x=  1 cur=  3 
  idx= 3 x= -6 cur= -3 
  idx= 4 x=  6 cur=  3 
  idx= 5 x=  7 cur= 10 
  idx= 6 x=  9 cur= 19 
  idx= 7 x= -1 cur= 18 
  idx= 8 x=  2 cur= 20 
  idx= 9 x=  0 cur= 20 
  idx=10 x=  1 cur= 21 
parts = 0 -> False


False

`target = 7`: tổng chạy đi `0, 2, 3, -3, 3, 10, 19, 18, 20, 20, 21` — **không lần nào** đúng bằng 7 (nhảy từ 3 lên 10) → `parts = 0` → `False`. Tổng chia hết cho 3 là điều kiện **cần**, chưa phải **đủ**.

### 7.2. Trace ca `target = 0` — tại sao cần `>= 3`

In [15]:
trace_greedy([0, 0, 0, 0])
print()
trace_greedy([1, -1, 1, -1])

arr = [0, 0, 0, 0] | total = 0
target = 0
  idx= 0 x=  0 cur=  0 == target -> cắt, parts=1
  idx= 1 x=  0 cur=  0 == target -> cắt, parts=2
  idx= 2 x=  0 cur=  0 == target -> cắt, parts=3
  idx= 3 x=  0 cur=  0 == target -> cắt, parts=4
parts = 4 -> True

arr = [1, -1, 1, -1] | total = 0
target = 0
  idx= 0 x=  1 cur=  1 
  idx= 1 x= -1 cur=  0 == target -> cắt, parts=1
  idx= 2 x=  1 cur=  1 
  idx= 3 x= -1 cur=  0 == target -> cắt, parts=2
parts = 2 -> False


False

`[1,-1,1,-1]`: tổng 0, nhưng chỉ chạm 0 hai lần (idx 1 và idx 3) → `False`. Thử tay: không có cách cắt 3 đoạn tổng 0 vì mọi đoạn 1 phần tử đều khác 0, và các đoạn `[1,-1]`, `[-1,1]` chiếm 2 phần tử → không đủ chỗ cho 3 đoạn.

## 8. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def canThreePartsEqualSum(self, arr: list[int]) -> bool:
        total = sum(arr)
        if total % 3 != 0:
            return False
        target = total // 3
        cur = parts = 0
        for x in arr:
            cur += x
            if cur == target:
                parts += 1
                cur = 0
        return parts >= 3

sol = Solution()
assert sol.canThreePartsEqualSum([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]) is True
assert sol.canThreePartsEqualSum([0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1]) is False
assert sol.canThreePartsEqualSum([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]) is True
print("Solution PASS 3 ví dụ của đề")

Solution PASS 3 ví dụ của đề


Phiên bản dừng sớm (có thể `return True` ngay khi `parts == 2` **và còn phần tử phía sau**):

In [17]:
class SolutionEarlyStop:
    def canThreePartsEqualSum(self, arr: list[int]) -> bool:
        total = sum(arr)
        if total % 3 != 0:
            return False
        target = total // 3
        cur = parts = 0
        for idx in range(len(arr) - 1):        # chừa phần tử cuối cho phần 3
            cur += arr[idx]
            if cur == target:
                parts += 1
                cur = 0
                if parts == 2:
                    return True
        return False

assert SolutionEarlyStop().canThreePartsEqualSum([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]) is True
assert SolutionEarlyStop().canThreePartsEqualSum([1, -1, 1, -1]) is False
print("SolutionEarlyStop PASS")

SolutionEarlyStop PASS


## 9. Vì sao đúng?

**(⇒) Nếu thuật toán trả `True`:** ta có ít nhất 3 lần cắt, mỗi đoạn giữa hai lần cắt có tổng `target` và khác rỗng (phải cộng ít nhất một phần tử thì `cur` mới chạm `target` lần nữa, vì `cur` được reset về 0 — ngay cả khi `target = 0` thì lần chạm tiếp theo cũng đòi ít nhất một phần tử được cộng). Lấy đoạn 1, đoạn 2, và **gộp mọi thứ còn lại** thành đoạn 3: tổng đoạn 3 = `total - 2·target = target`. Đoạn 3 chứa ít nhất đoạn thứ ba đã đếm nên khác rỗng.

**(⇐) Nếu tồn tại cách chia:** theo mục 5.1, cắt sớm nhất không làm mất lời giải → thuật toán tìm được 2 lần cắt đầu tiên với phần còn lại khác rỗng có tổng `target`. Tổng chạy trên phần còn lại cuối cùng sẽ chạm `target` ít nhất một lần (ở phần tử cuối) → `parts ≥ 3`.

**Bất biến:** tại mỗi bước, `cur` = tổng các phần tử kể từ lần cắt gần nhất; `parts` = số đoạn liên tiếp đã tìm, mỗi đoạn tổng `target`.

## 10. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---|---|
| 1. Thử cặp + `sum` | mọi `(i, j)`, cộng lại | `O(n³)` | `O(n)` (slicing tạo list) |
| 2. Thử cặp + prefix | hiệu prefix | `O(n²)` | `O(n)` |
| 3. Tham lam đếm | quét một lần | `O(n)` | `O(1)` |
| 3b. Hai điểm cắt | tìm `t` rồi `2t` | `O(n)` | `O(1)` |
| 4. Hai con trỏ | trái tìm `t`, phải tìm `t` | `O(n)` | `O(1)` |

`sum(arr)` tốn O(n) một lần; vòng quét O(n) → tổng O(n).

## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<40} PASS ({actual})")

cases = [
    ("đề 1",                                   [0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1], True),
    ("đề 2",                                   [0, 2, 1, -6, 6, 7, 9, -1, 2, 0, 1], False),
    ("đề 3",                                   [3, 3, 6, 5, -2, 2, 5, 1, -9, 4], True),
    ("n = 3 bằng nhau",                        [1, 1, 1], True),
    ("n = 3 không bằng",                       [1, 2, 3], False),
    ("tổng không chia hết 3",                  [1, 1, 1, 1], False),
    ("target = 0, nhiều số 0",                 [0, 0, 0, 0], True),
    ("target = 0, chỉ chạm 2 lần",             [1, -1, 1, -1], False),
    ("phần 3 rỗng nếu không cẩn thận",         [1, -1, 0], False),
    ("2t xuất hiện trước t",                   [2, -1, 2], False),
    ("tổng âm",                                [-1, -1, -1], True),
    ("target 0 hợp lệ có số khác 0",           [1, -1, 0, 2, -2], True),
]
solvers = {
    "Solution": lambda a: Solution().canThreePartsEqualSum(a),
    "EarlyStop": lambda a: SolutionEarlyStop().canThreePartsEqualSum(a),
    "two_cuts": can_three_two_cuts,
    "two_pointers": can_three_two_pointers,
    "prefix_pairs": can_three_prefix_pairs,
    "bruteforce": can_three_bruteforce,
}
for name, arr, exp in cases:
    for sname, f in solvers.items():
        assert f(arr) == exp, (name, sname, arr)
    check(name, Solution().canThreePartsEqualSum(arr), exp)

đề 1                                     PASS (True)
đề 2                                     PASS (False)
đề 3                                     PASS (True)
n = 3 bằng nhau                          PASS (True)
n = 3 không bằng                         PASS (False)
tổng không chia hết 3                    PASS (False)
target = 0, nhiều số 0                   PASS (True)
target = 0, chỉ chạm 2 lần               PASS (False)
phần 3 rỗng nếu không cẩn thận           PASS (False)
2t xuất hiện trước t                     PASS (False)
tổng âm                                  PASS (True)
target 0 hợp lệ có số khác 0             PASS (True)


### 11.1. Đối chiếu ngẫu nhiên với brute force (có seed)

Chọn giá trị nhỏ trong `[-2, 2]` để có nhiều ca `True` và nhiều ca `target = 0`.

In [19]:
import random
random.seed(1013)
count_true = 0
for _ in range(3000):
    n = random.randint(3, 9)
    arr = [random.randint(-2, 2) for _ in range(n)]
    expected = can_three_bruteforce(arr)
    count_true += expected
    for sname, f in solvers.items():
        assert f(arr) == expected, (sname, arr, expected)
print(f"3000 mảng ngẫu nhiên: mọi cách khớp brute force ({count_true} ca True)")

3000 mảng ngẫu nhiên: mọi cách khớp brute force (367 ca True)


### 11.2. Input lớn: các cách O(n) chạy nhanh

In [20]:
import time
random.seed(5)
big = [random.randint(-10**4, 10**4) for _ in range(50_000)]
# ép một lời giải: điều chỉnh để ba đoạn có tổng bằng nhau
s1, s2, s3 = sum(big[:10000]), sum(big[10000:30000]), sum(big[30000:])
big[29999] += s1 - s2
big[-1] += s1 - s3
start = time.perf_counter()
res = [f(big) for sname, f in solvers.items() if sname not in ("bruteforce", "prefix_pairs")]
print(res, f"| {time.perf_counter() - start:.3f}s cho 4 cách O(n)")

[True, True, True, True] | 0.008s cho 4 cách O(n)


Kết quả đều `True`. Ta không chạy hai cách O(n²)/O(n³) trên input này vì sẽ mất rất lâu.

## 12. Sai lầm người mới hay mắc

### 12.1. Chỉ kiểm tra "có prefix = t và có prefix = 2t" mà quên thứ tự

In [21]:
def wrong_no_order(arr):
    total = sum(arr)
    if total % 3:
        return False
    t = total // 3
    pre = list(accumulate(arr))
    return t in pre and 2 * t in pre

print("sai :", wrong_no_order([2, -1, 2]))
print("đúng:", Solution().canThreePartsEqualSum([2, -1, 2]))

sai : True
đúng: False


**Vì sao sai:** `[2,-1,2]`: prefix `[2,1,3]`, `t = 1`, `2t = 2`. Có cả hai giá trị, nhưng `2t` xuất hiện **trước** `t` → không phải cách cắt hợp lệ. **Sửa:** tìm `2t` **sau** vị trí của `t`.

### 12.2. Để phần 3 rỗng

In [22]:
def wrong_empty_third(arr):
    total = sum(arr)
    if total % 3:
        return False
    t = total // 3
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == t:
            parts += 1
            cur = 0
            if parts == 2:
                return True      # SAI: chưa chắc còn phần tử cho phần 3
    return False

print("sai :", wrong_empty_third([1, -1, 0]))
print("đúng:", Solution().canThreePartsEqualSum([1, -1, 0]))

sai : True
đúng: False


**Vì sao sai:** `[1,-1,0]`: chạm 0 tại idx 1 và idx 2 (phần tử cuối) → phần 3 rỗng. **Sửa:** chỉ duyệt tới `n - 2` khi dừng sớm (như `SolutionEarlyStop`), hoặc đếm hết rồi dùng `parts >= 3`.

### 12.3. Dùng `== 3` thay vì `>= 3`

In [23]:
def wrong_eq3(arr):
    total = sum(arr)
    if total % 3:
        return False
    t = total // 3
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == t:
            parts += 1
            cur = 0
    return parts == 3

print("sai :", wrong_eq3([0, 0, 0, 0]))
print("đúng:", Solution().canThreePartsEqualSum([0, 0, 0, 0]))

sai : False
đúng: True


**Vì sao sai:** với `target = 0` có thể đếm được nhiều hơn 3 đoạn. **Sửa:** `parts >= 3`.

### 12.4. Dùng `/` rồi so sánh với tổng nguyên

`target = total / 3` cho `float`. Với số lớn hoặc tổng không chia hết, so sánh `int == float` dễ gây nhầm (ví dụ `total = 10` → `3.333...`, không bao giờ chạm nhưng code vẫn chạy mà không báo `False` sớm). **Sửa:** kiểm tra `total % 3 == 0` trước, rồi dùng `//`.

### 12.5. Sắp xếp mảng trước

Các phần phải là **đoạn liên tiếp theo thứ tự gốc**. Sắp xếp làm thay đổi bài toán.

In [24]:
arr = [1, -1, 0, 0]          # gốc: [1,-1],[0],[0] -> True
print("gốc      :", arr, "->", Solution().canThreePartsEqualSum(arr))
print("sau sort :", sorted(arr), "->", Solution().canThreePartsEqualSum(sorted(arr)))

gốc      : [1, -1, 0, 0] -> True
sau sort : [-1, 0, 0, 1] -> False


`[1,-1,0,0]` chia được thành `[1,-1]`, `[0]`, `[0]`. Sau khi sắp xếp thành `[-1,0,0,1]` thì không có đoạn đầu nào có tổng 0 (tổng chạy là `-1,-1,-1,0`, chỉ chạm 0 ở cuối) → `False`. Kết luận: **không được sắp xếp**.

## 13. Nhận diện pattern

Dấu hiệu:
- "chia mảng thành các **đoạn liên tiếp** có tổng bằng nhau" → tổng mỗi đoạn biết trước = `total / k`.
- "tổng đoạn con" → nghĩ tới **prefix sum**.
- Khi biết trước tổng mục tiêu → **tham lam cắt sớm nhất** thường đúng.

Bài liên quan:
- LeetCode 724 (Find Pivot Index), 1991 (Find the Middle Index) — prefix sum chia 2 phần.
- LeetCode 560 (Subarray Sum Equals K) — prefix sum + hash map.
- LeetCode 698 (Partition to K Equal Sum Subsets) — **tập con** (không liên tiếp) → khó hơn nhiều (backtracking).
- LeetCode 1712 (Ways to Split Array Into Three Subarrays) — đếm số cách chia 3 đoạn với ràng buộc.

## 14. Biến thể

### 14.1. Biến thể A — chia thành `k` đoạn liên tiếp tổng bằng nhau

In [25]:
def can_k_parts(arr: list[int], k: int) -> bool:
    total = sum(arr)
    if total % k != 0:
        return False
    target = total // k
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == target:
            parts += 1
            cur = 0
    return parts >= k

print(can_k_parts([1, 1, 1, 1], 4), can_k_parts([1, 1, 1, 1], 2), can_k_parts([1, 2, 1], 2))

True True False


### 14.2. Biến thể B — trả về **chỉ số cắt** `[i, j]` (theo định nghĩa của đề) hoặc `[-1, -1]`

In [26]:
def three_parts_indices(arr: list[int]) -> list[int]:
    total = sum(arr)
    if total % 3:
        return [-1, -1]
    t = total // 3
    n = len(arr)
    s = 0
    i = -1
    for idx in range(n):
        s += arr[idx]
        if s == t:
            i = idx
            break
    if i == -1:
        return [-1, -1]
    for idx in range(i + 1, n - 1):
        s += arr[idx]
        if s == 2 * t:
            return [i, idx + 1]      # phần 3 bắt đầu tại j = idx + 1
    return [-1, -1]

print(three_parts_indices([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]))   # [2, 8]
print(three_parts_indices([3, 3, 6, 5, -2, 2, 5, 1, -9, 4]))      # [1, 3]

[2, 8]
[1, 3]


## 15. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `arr = [1, 3, -1, 2, 1, 0, 3]`, tính `total`, `target`, lập bảng `cur`/`parts` như mục 7 và cho đáp án.

### Bài 2 — Dự đoán output
Đoán `Solution().canThreePartsEqualSum([0, 0, 0])` và `Solution().canThreePartsEqualSum([1, -1, 1, -1, 0])`.

### Bài 3 — TODO: viết lại lời giải tham lam

In [27]:
def exercise_three_parts(arr: list[int]) -> bool:
    # TODO 1: tổng và kiểm tra chia hết
    # TODO 2: quét, đếm số lần cur chạm target (reset cur)
    # TODO 3: trả parts >= 3
    pass

# assert exercise_three_parts([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]) is True
# assert exercise_three_parts([1, -1, 1, -1]) is False

### Bài 4 — TODO: đếm **số cách** chia thành 3 đoạn tổng bằng nhau

Đếm số cặp `(i, j)` hợp lệ. Ví dụ `[0,0,0,0]` có 3 cách. Yêu cầu O(n).

In [28]:
def exercise_count_ways(arr: list[int]) -> int:
    # TODO: mỗi vị trí kết thúc phần 2 có prefix = 2t, cộng số vị trí kết thúc phần 1 (prefix = t) đứng trước nó
    pass

# assert exercise_count_ways([0, 0, 0, 0]) == 3
# assert exercise_count_ways([1, 2, 3, 0, 3]) == 2
# assert exercise_count_ways([1, 1, 1]) == 1

### Bài 5 — TODO biến thể: chia thành **2** đoạn liên tiếp tổng bằng nhau (trả về `True/False`)

In [29]:
def exercise_two_parts(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_two_parts([1, 2, 3]) is True
# assert exercise_two_parts([1, 1, 1]) is False
# assert exercise_two_parts([0, 0]) is True

### Bài 6 — Mini challenge: tìm phản ví dụ cho "sắp xếp trước"

Viết hàm `exercise_find_sort_counterexample()` thử ngẫu nhiên (có seed) các mảng nhỏ và trả về **một** mảng mà đáp án gốc khác đáp án sau khi `sorted`.

In [30]:
def exercise_find_sort_counterexample() -> list[int]:
    # TODO
    pass

# arr = exercise_find_sort_counterexample()
# assert Solution().canThreePartsEqualSum(arr) != Solution().canThreePartsEqualSum(sorted(arr))

## 16. Gợi ý ba mức

**Bài 1** — (1) total = 9; (2) target = 3; (3) tổng chạy bắt đầu `1, 4, …` — lần đầu chạm 3 là khi cộng tới `-1` ở idx 2.

**Bài 2** — (1) target = 0; (2) đếm số lần `cur` chạm 0; (3) `[1,-1,1,-1,0]` chạm 0 ở idx 1, 3, 4.

**Bài 3** — (1) mục 6; (2) `cur = parts = 0`; (3) `if cur == target: parts += 1; cur = 0`.

**Bài 4** — (1) dùng prefix tiền tố `pre[k]` = tổng `arr[0..k]`; (2) phần 1 kết thúc tại `a` (`pre[a] = t`), phần 2 kết thúc tại `b` (`pre[b] = 2t`), cần `a < b < n - 1`; (3) quét `b` từ trái sang, giữ biến `cnt_t` = số `a < b` có `pre[a] = t`; **cập nhật `cnt_t` sau khi xử lý `b`**.

**Bài 5** — (1) `total % 2`; (2) prefix chạm `total // 2` tại vị trí `< n - 1`; (3) dùng `range(len(arr) - 1)`.

**Bài 6** — (1) `random.seed(...)`; (2) mảng 3–6 phần tử trong `[-3, 3]`; (3) lặp tới khi tìm được rồi `return`.

## 17. Đáp án tham khảo

**Bài 1:** `total = 9`, `target = 3`. `cur`: 1, 4, 3 ✓(parts=1, reset), 2, 3 ✓(parts=2, reset), 0, 3 ✓(parts=3) → `True`. Cách chia: `[1,3,-1]`, `[2,1]`, `[0,3]`.

**Bài 2:** `[0,0,0]` → `True` (3 lần chạm 0). `[1,-1,1,-1,0]` → `True` (chạm 3 lần: `[1,-1]`, `[1,-1]`, `[0]`).

In [31]:
def answer_three_parts(arr):
    total = sum(arr)
    if total % 3:
        return False
    target = total // 3
    cur = parts = 0
    for x in arr:
        cur += x
        if cur == target:
            parts += 1
            cur = 0
    return parts >= 3


def answer_count_ways(arr):
    total = sum(arr)
    if total % 3:
        return 0
    t = total // 3
    n = len(arr)
    pre = 0
    cnt_t = 0
    ways = 0
    for b_idx in range(n - 1):          # b_idx: vị trí kết thúc phần 2, phải < n - 1
        pre += arr[b_idx]
        if pre == 2 * t:
            ways += cnt_t               # số a < b_idx với pre[a] = t
        if pre == t:
            cnt_t += 1                  # cập nhật SAU để đảm bảo a < b_idx
    return ways


def answer_two_parts(arr):
    total = sum(arr)
    if total % 2:
        return False
    half = total // 2
    s = 0
    for idx in range(len(arr) - 1):
        s += arr[idx]
        if s == half:
            return True
    return False


def answer_find_sort_counterexample():
    rnd = random.Random(99)
    while True:
        arr = [rnd.randint(-3, 3) for _ in range(rnd.randint(3, 6))]
        if answer_three_parts(arr) != answer_three_parts(sorted(arr)):
            return arr


def count_ways_bruteforce(arr):
    n = len(arr)
    return sum(1 for i in range(n) for j in range(i + 2, n)
               if sum(arr[:i + 1]) == sum(arr[i + 1:j]) == sum(arr[j:]))


assert answer_three_parts([1, 3, -1, 2, 1, 0, 3]) is True
assert answer_three_parts([0, 0, 0]) is True and answer_three_parts([1, -1, 1, -1, 0]) is True
assert answer_three_parts([0, 2, 1, -6, 6, -7, 9, 1, 2, 0, 1]) is True
assert answer_three_parts([1, -1, 1, -1]) is False
assert answer_count_ways([0, 0, 0, 0]) == 3
assert answer_count_ways([1, 2, 3, 0, 3]) == 2
assert answer_count_ways([1, 1, 1]) == 1
assert answer_two_parts([1, 2, 3]) is True and answer_two_parts([1, 1, 1]) is False and answer_two_parts([0, 0]) is True
ce = answer_find_sort_counterexample()
assert answer_three_parts(ce) != answer_three_parts(sorted(ce))
print("Phản ví dụ cho 'sắp xếp trước':", ce, "->", answer_three_parts(ce), "| sorted:", sorted(ce), "->", answer_three_parts(sorted(ce)))

random.seed(77)
for _ in range(2000):
    arr = [random.randint(-2, 2) for _ in range(random.randint(3, 8))]
    assert answer_count_ways(arr) == count_ways_bruteforce(arr), arr
print("Đáp án tham khảo PASS toàn bộ test!")

Phản ví dụ cho 'sắp xếp trước': [-2, 0, -1, -2, -1, -3] -> True | sorted: [-3, -2, -2, -1, -1, 0] -> False


Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 4:** với mỗi vị trí `b` kết thúc phần 2 (prefix = `2t`, `b < n - 1`), mọi vị trí `a < b` kết thúc phần 1 (prefix = `t`) đều tạo một cách chia. Cập nhật `cnt_t` **sau** khi xét `b` để không đếm `a = b` (phần 2 rỗng) — quan trọng khi `t = 0` vì khi đó `t = 2t`.

`random.Random(99)` tạo một bộ sinh số ngẫu nhiên **riêng** (không ảnh hưởng `random.seed` toàn cục).

## 18. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def canThreePartsEqualSum(self, arr: list[int]) -> bool:
        total = sum(arr)
        if total % 3 != 0:
            return False
        target = total // 3
        cur = parts = 0
        for x in arr:
            cur += x
            if cur == target:
                parts += 1
                cur = 0
        return parts >= 3
```

**Câu thần chú:** *"Tổng chia ba — cộng dồn chạm đích thì cắt — đủ ba lần là xong."*

**Checklist trước khi submit:**
- [ ] Kiểm tra `total % 3 != 0` → `False`?
- [ ] Dùng `//` (không phải `/`)?
- [ ] Không sắp xếp mảng (các phần là đoạn liên tiếp)?
- [ ] Thứ tự: điểm cắt `2t` phải **sau** điểm cắt `t`?
- [ ] Phần 3 khác rỗng (nếu dừng sớm, chỉ duyệt tới `n - 2`)?
- [ ] `parts >= 3` (không phải `== 3`) để xử lý `target = 0`?
- [ ] Đã test `[0,0,0,0]`, `[1,-1,1,-1]`, `[1,-1,0]`, `[2,-1,2]`?